# Module 4.2: BERT 架构 (BERT Architecture)

## 1. 本章概览

### 📚 学习目标

1. **双向编码**：理解 BERT 的创新 - 双向预训练
2. **MLM**：掌握掩码语言建模目标
3. **NSP**：理解下一句预测任务
4. **微调**：学习如何将 BERT 应用于下游任务

### 🎯 核心问题

- 为什么双向上下文很重要？
- BERT 如何通过 MLM 实现双向编码？
- BERT 与 GPT 有什么区别？

### 🏢 业务场景

本章技术将应用于 `电商客服智能助理` 场景：
- 客服意图分类如何快速提升准确率？→ BERT 预训练 + 微调范式
- 为什么需要双向上下文理解？→ MLM 任务的设计动机

### ⏱️ 预计学习时间：4-5小时

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
torch.manual_seed(42)

print("✓ Libraries imported")

## 2. 为什么需要 BERT？

### 2.1 预训练模型的动机

**传统 NLP 的困境**：

```
任务 A (情感分析)     任务 B (问答)        任务 C (NER)
    ↓                    ↓                   ↓
从头训练模型          从头训练模型         从头训练模型
    ↓                    ↓                   ↓
需要大量标注数据      需要大量标注数据     需要大量标注数据
```

**问题**：
- 每个任务都需要大量标注数据（昂贵！）
- 无法利用海量无标注文本
- 模型无法共享语言知识

**预训练模型的解决方案**：

```
大规模无标注文本 (Wikipedia, Books, Web)
            ↓
    预训练通用语言模型
            ↓
    ┌───────┼───────┐
    ↓       ↓       ↓
  任务A    任务B    任务C
  (微调)  (微调)  (微调)
```

**优势**：
- ✅ 利用海量无标注数据学习语言知识
- ✅ 少量标注数据即可微调
- ✅ 多个任务共享预训练知识

### 2.2 单向模型的局限性

**ELMo 和 GPT 的问题**：

在 BERT 之前，预训练模型主要是单向的：

**GPT (单向，左到右)**：
$$P(w_1, w_2, ..., w_n) = \prod_{i=1}^{n} P(w_i | w_1, ..., w_{i-1})$$

**问题示例**：

```
句子: "The bank of the river is beautiful."

单向模型处理 "bank":
  已知: "The"
  未知: "of the river is beautiful"
  → 无法确定是 "银行" 还是 "河岸"

双向模型处理 "bank":
  已知: "The" + "of the river is beautiful"
  → 明确知道是 "河岸"
```

### 2.3 双向编码的价值

**为什么双向很重要？**

1. **歧义消解**：利用上下文消除词义歧义
2. **长距离依赖**：捕获前后文的关联
3. **语义理解**：更深入理解句子含义

**实际任务中的优势**：

| 任务 | 单向模型 | 双向模型 |
|------|----------|----------|
| 情感分析 | 只看前文 | 看完整句子 |
| 问答 | 难以定位答案 | 精确定位 |
| NER | 上下文不足 | 充分上下文 |
| 文本分类 | 信息不完整 | 完整信息 |

In [ ]:
# 🔬 Micro Practice: 单向 vs 双向对比实验

def simulate_context_understanding(sentence, target_word_idx, use_bidirectional=False):
    """
    模拟单向和双向模型的上下文理解
    
    Args:
        sentence: 句子（单词列表）
        target_word_idx: 目标词的索引
        use_bidirectional: 是否使用双向上下文
    """
    if use_bidirectional:
        # 双向：看到所有上下文
        left_context = sentence[:target_word_idx]
        right_context = sentence[target_word_idx+1:]
        context = left_context + ['[TARGET]'] + right_context
        context_type = "双向上下文"
    else:
        # 单向：只看左侧上下文
        left_context = sentence[:target_word_idx]
        context = left_context + ['[TARGET]'] + ['?'] * (len(sentence) - target_word_idx - 1)
        context_type = "单向上下文（左到右）"
    
    return context, context_type

# 测试歧义词
test_cases = [
    {
        'sentence': ['The', 'bank', 'of', 'the', 'river', 'is', 'beautiful'],
        'target_idx': 1,
        'target_word': 'bank',
        'meanings': ['银行', '河岸']
    },
    {
        'sentence': ['I', 'went', 'to', 'the', 'bank', 'to', 'deposit', 'money'],
        'target_idx': 4,
        'target_word': 'bank',
        'meanings': ['银行', '河岸']
    },
    {
        'sentence': ['The', 'bat', 'flew', 'out', 'of', 'the', 'cave'],
        'target_idx': 1,
        'target_word': 'bat',
        'meanings': ['蝙蝠', '球棒']
    }
]

print("单向 vs 双向上下文对比")
print("=" * 80)

for i, case in enumerate(test_cases, 1):
    print(f"\n示例 {i}: 目标词 '{case['target_word']}' (可能含义: {', '.join(case['meanings'])})")
    print("-" * 80)
    
    # 单向上下文
    uni_context, uni_type = simulate_context_understanding(
        case['sentence'], case['target_idx'], use_bidirectional=False
    )
    print(f"\n{uni_type}:")
    print(f"  {' '.join(uni_context)}")
    print(f"  → 信息不足，难以消歧")
    
    # 双向上下文
    bi_context, bi_type = simulate_context_understanding(
        case['sentence'], case['target_idx'], use_bidirectional=True
    )
    print(f"\n{bi_type}:")
    print(f"  {' '.join(bi_context)}")
    print(f"  → 完整上下文，可以准确消歧")

print("\n" + "=" * 80)
print("\n关键观察:")
print("  1. 单向模型在处理目标词时看不到后续上下文")
print("  2. 双向模型可以利用完整句子信息")
print("  3. 对于歧义词，双向上下文至关重要")
print("\n✓ 单向 vs 双向对比完成！")

In [ ]:
# 🔬 Micro Practice: 不同任务中双向上下文的优势

import matplotlib.pyplot as plt
import numpy as np

def visualize_bidirectional_advantage():
    """
    可视化双向模型在不同任务上的优势
    """
    tasks = ['情感分析', '问答系统', '命名实体\n识别', '文本分类', '语义\n相似度']
    
    # 模拟性能数据（相对提升百分比）
    unidirectional = [75, 68, 72, 78, 70]  # 单向模型性能
    bidirectional = [88, 92, 89, 90, 91]   # 双向模型性能
    
    x = np.arange(len(tasks))
    width = 0.35
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 性能对比
    bars1 = ax1.bar(x - width/2, unidirectional, width, label='单向模型 (GPT-style)', 
                    color='#e74c3c', alpha=0.8)
    bars2 = ax1.bar(x + width/2, bidirectional, width, label='双向模型 (BERT-style)', 
                    color='#2ecc71', alpha=0.8)
    
    ax1.set_xlabel('任务类型', fontsize=12)
    ax1.set_ylabel('性能得分', fontsize=12)
    ax1.set_title('单向 vs 双向模型性能对比', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(tasks)
    ax1.legend()
    ax1.set_ylim([0, 100])
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 添加数值标签
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9)
    
    # 相对提升
    improvements = [(b - u) / u * 100 for u, b in zip(unidirectional, bidirectional)]
    colors = ['#3498db' if imp > 15 else '#95a5a6' for imp in improvements]
    
    bars3 = ax2.bar(tasks, improvements, color=colors, alpha=0.8)
    ax2.set_xlabel('任务类型', fontsize=12)
    ax2.set_ylabel('相对提升 (%)', fontsize=12)
    ax2.set_title('双向模型的相对提升', fontsize=14, fontweight='bold')
    ax2.axhline(y=15, color='r', linestyle='--', alpha=0.5, label='显著提升阈值')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 添加数值标签
    for bar in bars3:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%',
                ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("\n关键发现:")
    print("  1. 问答系统受益最大 (+35.3%) - 需要精确定位答案")
    print("  2. 语义相似度提升显著 (+30.0%) - 需要理解完整语义")
    print("  3. NER 任务提升明显 (+23.6%) - 需要充分上下文")
    print("  4. 所有理解类任务都显著受益于双向编码")

visualize_bidirectional_advantage()
print("\n✓ 任务对比可视化完成！")

## 2. 双向上下文的重要性

### 2.1 单向 vs 双向

**传统语言模型（如 GPT）**：
- 自回归：$P(w_i | w_{<i})$ - 只能看到左侧上下文
- 适合生成任务

**掩码语言建模 (Masked Language Modeling, MLM)** 是 BERT 的核心预训练方式，通过随机遮盖输入中的部分 token 并让模型预测被遮盖的内容，从而实现双向上下文编码。在本章场景中，它是 BERT 区别于 GPT 等单向模型的关键设计。

**下一句预测 (Next Sentence Prediction, NSP)** 是 BERT 的另一个预训练目标，判断两个句子是否在原文中相邻，帮助模型学习句子间的关系。在本章场景中，它与 MLM 共同构成 BERT 的预训练任务。

**BERT（双向）**：
- 掩码语言建模：$P(w_i | w_{<i}, w_{>i})$ - 可以看到两侧上下文
- 适合理解任务

### 2.2 为什么双向很重要？

考虑句子："The bank of the river"
- 单向模型在看到 "bank" 时不知道是银行还是河岸
- 双向模型可以利用 "river" 来消歧

### 2.3 BERT vs GPT

| 特性 | BERT | GPT |
|------|------|-----|
| 方向 | 双向 | 单向（左到右）|
| 预训练 | MLM + NSP | 自回归 LM |
| 适用任务 | 理解（分类、QA）| 生成 |
| 架构 | Encoder only | Decoder only |

## 4. 掩码语言建模 (Masked Language Modeling)

### 4.1 MLM 目标

随机掩盖输入中 15% 的 tokens，然后预测被掩盖的 tokens。

**掩码策略**：
- 80%：替换为 [MASK]
- 10%：替换为随机 token
- 10%：保持原样

**为什么不全部用 [MASK]？**
- 微调时没有 [MASK] token
- 随机替换和保持原样增加鲁棒性

### 4.2 MLM 损失函数完整推导

**目标**：最大化被掩盖 tokens 的条件概率

给定句子 $\mathbf{x} = [x_1, x_2, ..., x_n]$，掩码位置集合 $\mathcal{M}$：

$$\mathcal{L}_{\text{MLM}} = -\frac{1}{|\mathcal{M}|} \sum_{i \in \mathcal{M}} \log P(x_i | \mathbf{x}_{\backslash \mathcal{M}})$$

其中 $\mathbf{x}_{\backslash \mathcal{M}}$ 表示除掩码位置外的所有 tokens。

**详细步骤**：

1. **编码器输出**：
   $$\mathbf{h}_i = \text{Encoder}(\mathbf{x}_{\backslash \mathcal{M}})_i$$

2. **预测分布**：
   $$P(x_i | \mathbf{x}_{\backslash \mathcal{M}}) = \text{softmax}(\mathbf{W}_{\text{vocab}} \mathbf{h}_i + \mathbf{b})$$

3. **交叉熵损失**：
   $$\mathcal{L}_i = -\log P(x_i^{\text{true}} | \mathbf{x}_{\backslash \mathcal{M}})$$

### 4.3 为什么使用 80-10-10 策略？

**实验对比**：

| 策略 | 预训练困惑度 | 微调性能 | 问题 |
|------|-------------|----------|------|
| 100% [MASK] | 低 | 中 | 预训练-微调不匹配 |
| 80-10-10 | 中 | 高 | 平衡性能和鲁棒性 |
| 随机替换 | 高 | 低 | 任务过难 |

**80-10-10 的优势**：
1. **80% [MASK]**：主要学习目标，明确的预测任务
2. **10% 随机**：增加鲁棒性，防止过拟合 [MASK]
3. **10% 保持**：学习上下文表示，即使没有明显错误

### 4.4 梯度流分析

**反向传播路径**：

```
损失 L_MLM
    ↓
Softmax 层
    ↓
MLM Head (线性层)
    ↓
Encoder 输出 h_i
    ↓
Transformer 层 N
    ↓
...
    ↓
Transformer 层 1
    ↓
Embedding 层
```

**关键特性**：
- 只有被掩盖的位置产生梯度
- 梯度通过 self-attention 传播到所有位置
- 双向信息流确保充分学习

In [ ]:
# 🔬 Micro Practice: MLM Data Preparation

def create_mlm_data(tokens, mask_prob=0.15, vocab_size=1000):
    """
    Create MLM training data
    
    Args:
        tokens: List of token ids
        mask_prob: Probability of masking (default 0.15)
        vocab_size: Size of vocabulary
    
    Returns:
        masked_tokens: Tokens with masking applied
        labels: Original tokens (for loss computation)
        mask_positions: Boolean mask of masked positions
    """
    tokens = np.array(tokens)
    labels = tokens.copy()
    
    # Randomly select positions to mask
    mask_positions = np.random.rand(len(tokens)) < mask_prob
    
    # Apply masking strategy
    for i in np.where(mask_positions)[0]:
        rand = np.random.rand()
        if rand < 0.8:
            tokens[i] = vocab_size  # [MASK] token
        elif rand < 0.9:
            tokens[i] = np.random.randint(0, vocab_size)  # Random token
        # else: keep original (10%)
    
    return tokens, labels, mask_positions

# Test
original = [10, 20, 30, 40, 50, 60, 70, 80]
masked, labels, mask_pos = create_mlm_data(original, mask_prob=0.5, vocab_size=100)

print(f"Original: {original}")
print(f"Masked:   {masked}")
print(f"Labels:   {labels}")
print(f"Mask pos: {mask_pos}")
print("\n✓ MLM data preparation implemented!")

## 4. 下一句预测 (Next Sentence Prediction)

### 4.1 NSP 目标

给定两个句子 A 和 B，预测 B 是否是 A 的下一句。

**训练数据构造**：
- 50%：B 是 A 的实际下一句（IsNext）
- 50%：B 是语料库中的随机句子（NotNext）

### 4.2 为什么需要 NSP？

帮助模型理解句子间关系，对以下任务有帮助：
- 问答（QA）
- 自然语言推理（NLI）
- 句子对分类

### 4.3 NSP 损失

$$\mathcal{L}_{\text{NSP}} = -\log P(\text{IsNext} | [CLS])$$

使用 [CLS] token 的表示进行二分类。

In [ ]:
# 🔬 Micro Practice: NSP Data Preparation

def create_nsp_data(sentences):
    """
    Create NSP training data
    
    Args:
        sentences: List of sentences
    
    Returns:
        pairs: List of (sentence_a, sentence_b) tuples
        labels: List of labels (1=IsNext, 0=NotNext)
    """
    pairs = []
    labels = []
    
    for i in range(len(sentences) - 1):
        # 50% actual next sentence
        if np.random.rand() < 0.5:
            pairs.append((sentences[i], sentences[i+1]))
            labels.append(1)  # IsNext
        # 50% random sentence
        else:
            random_idx = np.random.randint(0, len(sentences))
            while random_idx == i or random_idx == i+1:
                random_idx = np.random.randint(0, len(sentences))
            pairs.append((sentences[i], sentences[random_idx]))
            labels.append(0)  # NotNext
    
    return pairs, labels

# Test
sentences = [
    "The cat sat on the mat.",
    "It was very comfortable.",
    "The dog ran in the park.",
    "Birds were singing."
]

pairs, labels = create_nsp_data(sentences)

print("NSP Training Examples:")
for (sent_a, sent_b), label in zip(pairs[:3], labels[:3]):
    print(f"\nSentence A: {sent_a}")
    print(f"Sentence B: {sent_b}")
    print(f"Label: {'IsNext' if label == 1 else 'NotNext'}")

print("\n✓ NSP data preparation implemented!")

### 5.4 NSP 深入分析

**NSP 的理论基础**：

句子关系建模对以下任务至关重要：
- **问答 (QA)**：问题和段落的关系
- **自然语言推理 (NLI)**：前提和假设的关系
- **对话系统**：上下文连贯性

**NSP 损失函数**：

$$\mathcal{L}_{\text{NSP}} = -\log P(\text{IsNext} | [\text{CLS}])$$

其中 $[\text{CLS}]$ token 的表示聚合了整个句子对的信息。

**为什么使用 [CLS] token？**

```
[CLS] Sentence A [SEP] Sentence B [SEP]
  ↓
通过 self-attention 聚合所有信息
  ↓
[CLS] 的最终表示包含句子对的全局信息
  ↓
二分类：IsNext / NotNext
```

### 5.5 NSP 的消融实验

**原始 BERT 论文的发现**：

| 模型 | MNLI | QNLI | SQuAD |
|------|------|------|-------|
| BERT (MLM + NSP) | 84.6 | 91.8 | 88.5 |
| BERT (仅 MLM) | 83.9 | 91.4 | 87.4 |

**提升**：NSP 带来 0.5-1.1% 的性能提升

### 5.6 RoBERTa 的发现

**RoBERTa 的实验**：

Liu et al. (2019) 发现：
- 移除 NSP 不会降低性能
- 甚至在某些任务上略有提升

**可能原因**：
1. **任务过于简单**：随机句子太容易识别
2. **主题混淆**：NSP 可能学到主题而非句子关系
3. **MLM 已足够**：MLM 本身已经学到句子级表示

**改进方向**：
- **SOP (Sentence Order Prediction)**：预测句子顺序
- **更难的负样本**：同主题的不相关句子

In [ ]:
# 🔬 Micro Practice: 完整 NSP 实现 — 数据准备、预测头与训练
# Goal: 实现端到端的 NSP 流水线，包括正负样本生成、基于 [CLS] 的二分类头、训练与评估
# Expected outcome: 理解 NSP 如何帮助 BERT 学习句子间关系

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

# ============================================================
# 1. NSP 数据准备：生成正负句子对
# ============================================================

def prepare_nsp_dataset(documents, num_samples=200):
    """
    从文档列表中生成 NSP 训练数据。
    
    正样本（IsNext）：从同一文档中取相邻句子对
    负样本（NotNext）：从不同文档中随机配对句子
    
    Args:
        documents: List[List[str]]，每个文档是一组句子
        num_samples: 生成样本总数
    Returns:
        pairs: List[(str, str)]
        labels: List[int]  1=IsNext, 0=NotNext
    """
    pairs, labels = [], []
    
    # 收集所有句子（用于生成负样本）
    all_sentences = [s for doc in documents for s in doc]
    
    for _ in range(num_samples):
        if np.random.rand() < 0.5:
            # --- 正样本：从同一文档取相邻句子 ---
            doc = documents[np.random.randint(len(documents))]
            if len(doc) < 2:
                continue
            idx = np.random.randint(len(doc) - 1)
            pairs.append((doc[idx], doc[idx + 1]))
            labels.append(1)
        else:
            # --- 负样本：随机配对不相关的句子 ---
            sent_a = all_sentences[np.random.randint(len(all_sentences))]
            sent_b = all_sentences[np.random.randint(len(all_sentences))]
            pairs.append((sent_a, sent_b))
            labels.append(0)
    
    return pairs, labels

# 示例文档集（模拟语料库）
documents = [
    [
        "BERT uses bidirectional encoding to understand context.",
        "This allows it to capture meaning from both directions.",
        "The result is a powerful language representation model."
    ],
    [
        "Machine learning requires large amounts of data.",
        "Deep learning models are especially data hungry.",
        "Transfer learning helps reduce data requirements."
    ],
    [
        "Natural language processing has many applications.",
        "Chatbots and translation are common use cases.",
        "Sentiment analysis is another important task."
    ],
]

pairs, labels = prepare_nsp_dataset(documents, num_samples=200)

# 统计正负样本数量
pos_count = sum(labels)
neg_count = len(labels) - pos_count
print(f"NSP 数据集统计：")
print(f"  总样本数: {len(labels)}")
print(f"  正样本 (IsNext):  {pos_count} ({pos_count/len(labels)*100:.1f}%)")
print(f"  负样本 (NotNext): {neg_count} ({neg_count/len(labels)*100:.1f}%)")
print(f"\n示例：")
for i in range(3):
    tag = "IsNext" if labels[i] == 1 else "NotNext"
    print(f"  [{tag}] A: {pairs[i][0][:50]}...")
    print(f"          B: {pairs[i][1][:50]}...")

# ============================================================
# 2. 简易 BERT + NSP 预测头
# ============================================================

class SimpleNSPModel(nn.Module):
    """
    简化版 BERT + NSP 头，用于教学演示。
    
    结构：
      Token Embedding + Position Embedding
        → Transformer Encoder (2 层)
        → 取 [CLS] 位置的输出
        → 二分类线性层 → IsNext / NotNext
    """
    
    def __init__(self, vocab_size=500, d_model=128, nhead=4, num_layers=2,
                 max_len=64, num_classes=2):
        super().__init__()
        self.d_model = d_model
        
        # Embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.seg_emb = nn.Embedding(2, d_model)          # Segment A / B
        self.pos_emb = nn.Embedding(max_len, d_model)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # NSP 预测头：基于 [CLS] token 的二分类
        self.nsp_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Tanh(),                      # BERT 原文使用 tanh 池化
            nn.Linear(d_model, num_classes)
        )
    
    def forward(self, input_ids, segment_ids):
        """
        Args:
            input_ids:   (batch, seq_len) 词元 ID
            segment_ids: (batch, seq_len) 段落 ID (0=句子A, 1=句子B)
        Returns:
            logits: (batch, 2)  IsNext / NotNext 的 logits
        """
        seq_len = input_ids.size(1)
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        
        # 三种 Embedding 相加（与 BERT 一致）
        x = self.token_emb(input_ids) + self.seg_emb(segment_ids) + self.pos_emb(positions)
        
        # Transformer 编码
        x = self.encoder(x)
        
        # 取 [CLS]（位置 0）的表示做 NSP 分类
        cls_output = x[:, 0, :]
        logits = self.nsp_head(cls_output)
        return logits

# ============================================================
# 3. 简单分词 + 数据编码
# ============================================================

# 构建一个简易词表（字符级别，仅用于演示）
CLS_TOKEN, SEP_TOKEN, PAD_TOKEN = 0, 1, 2
VOCAB_OFFSET = 3  # 实际词从 ID=3 开始

def simple_tokenize(text, max_len=20):
    """字符级分词（教学用途，实际应使用 WordPiece/BPE）"""
    return [ord(c) % 497 + VOCAB_OFFSET for c in text[:max_len]]

def encode_pair(sent_a, sent_b, max_len=60):
    """将句子对编码为 [CLS] A [SEP] B [SEP] 格式"""
    tokens_a = simple_tokenize(sent_a, max_len=25)
    tokens_b = simple_tokenize(sent_b, max_len=25)
    
    # [CLS] tokens_a [SEP] tokens_b [SEP]
    input_ids = [CLS_TOKEN] + tokens_a + [SEP_TOKEN] + tokens_b + [SEP_TOKEN]
    segment_ids = [0] * (len(tokens_a) + 2) + [1] * (len(tokens_b) + 1)
    
    # 填充到 max_len
    pad_len = max_len - len(input_ids)
    input_ids += [PAD_TOKEN] * pad_len
    segment_ids += [0] * pad_len
    
    return input_ids[:max_len], segment_ids[:max_len]

# 编码全部数据
all_input_ids, all_segment_ids, all_labels = [], [], []
for (sa, sb), label in zip(pairs, labels):
    ids, segs = encode_pair(sa, sb)
    all_input_ids.append(ids)
    all_segment_ids.append(segs)
    all_labels.append(label)

input_ids_t = torch.tensor(all_input_ids)
segment_ids_t = torch.tensor(all_segment_ids)
labels_t = torch.tensor(all_labels)

print(f"\n编码后数据形状：")
print(f"  input_ids:   {input_ids_t.shape}")
print(f"  segment_ids: {segment_ids_t.shape}")
print(f"  labels:      {labels_t.shape}")

# ============================================================
# 4. 训练 NSP 模型
# ============================================================

model = SimpleNSPModel(vocab_size=500, d_model=64, nhead=4, num_layers=2)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 简单的 mini-batch 训练
batch_size = 32
num_epochs = 10
n = len(labels_t)

print(f"\n开始训练 NSP 模型...")
print(f"  参数量: {sum(p.numel() for p in model.parameters()):,}")
print("=" * 50)

for epoch in range(num_epochs):
    # 随机打乱
    perm = torch.randperm(n)
    epoch_loss, correct = 0.0, 0
    
    for start in range(0, n, batch_size):
        idx = perm[start:start + batch_size]
        b_input = input_ids_t[idx]
        b_seg = segment_ids_t[idx]
        b_label = labels_t[idx]
        
        logits = model(b_input, b_seg)
        loss = criterion(logits, b_label)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * len(idx)
        correct += (logits.argmax(dim=-1) == b_label).sum().item()
    
    acc = correct / n * 100
    avg_loss = epoch_loss / n
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:2d}/{num_epochs}  Loss: {avg_loss:.4f}  Accuracy: {acc:.1f}%")

print("=" * 50)

# ============================================================
# 5. 评估：查看模型对新句子对的预测
# ============================================================

model.eval()
test_pairs = [
    ("BERT uses bidirectional encoding to understand context.",
     "This allows it to capture meaning from both directions.",
     "IsNext（相邻句子）"),
    ("Machine learning requires large amounts of data.",
     "Chatbots and translation are common use cases.",
     "NotNext（不同文档的句子）"),
]

print("\n评估结果：")
for sa, sb, desc in test_pairs:
    ids, segs = encode_pair(sa, sb)
    with torch.no_grad():
        logits = model(torch.tensor([ids]), torch.tensor([segs]))
        probs = torch.softmax(logits, dim=-1)[0]
    pred = "IsNext" if probs[1] > probs[0] else "NotNext"
    print(f"\n  期望: {desc}")
    print(f"  预测: {pred}  (IsNext={probs[1]:.3f}, NotNext={probs[0]:.3f})")

print("\n✓ NSP 完整实现完成：数据准备 → 模型定义 → 训练 → 评估")

In [ ]:
# 🔬 Micro Practice: MLM 损失函数可视化

import matplotlib.pyplot as plt
import numpy as np

def visualize_mlm_loss():
    """
    可视化 MLM 训练过程中的损失变化
    """
    # 模拟训练过程
    steps = np.arange(0, 10000, 100)

    # 不同掩码策略的损失曲线
    loss_100_mask = 8.0 * np.exp(-steps / 2000) + 2.0  # 100% [MASK]
    loss_80_10_10 = 8.5 * np.exp(-steps / 2000) + 1.8  # 80-10-10
    loss_random = 9.0 * np.exp(-steps / 2500) + 2.5    # 完全随机

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # 训练损失对比
    ax1.plot(steps, loss_100_mask, label='100% [MASK]', linewidth=2, color='#e74c3c')
    ax1.plot(steps, loss_80_10_10, label='80-10-10 策略', linewidth=2, color='#2ecc71')
    ax1.plot(steps, loss_random, label='完全随机替换', linewidth=2, color='#95a5a6')

    ax1.set_xlabel('训练步数', fontsize=12)
    ax1.set_ylabel('MLM 损失', fontsize=12)
    ax1.set_title('不同掩码策略的训练损失', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # 微调性能对比
    strategies = ['100%\n[MASK]', '80-10-10\n策略', '完全\n随机']
    pretrain_perf = [85, 88, 82]  # 预训练困惑度（越低越好，这里反转显示）
    finetune_perf = [86, 92, 84]  # 微调性能

    x = np.arange(len(strategies))
    width = 0.35

    bars1 = ax2.bar(x - width/2, pretrain_perf, width, label='预训练性能',
                    color='#3498db', alpha=0.8)
    bars2 = ax2.bar(x + width/2, finetune_perf, width, label='微调性能',
                    color='#2ecc71', alpha=0.8)

    ax2.set_xlabel('掩码策略', fontsize=12)
    ax2.set_ylabel('性能得分', fontsize=12)
    ax2.set_title('不同策略的性能对比', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(strategies)
    ax2.legend()
    ax2.set_ylim([0, 100])
    ax2.grid(True, alpha=0.3, axis='y')

    # 添加数值标签
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()

    print("\n关键发现:")
    print("  1. 80-10-10 策略在微调性能上最优")
    print("  2. 100% [MASK] 存在预训练-微调不匹配问题")
    print("  3. 完全随机替换任务过难，学习效率低")
    print("  4. 平衡策略是关键")

visualize_mlm_loss()
print("\n✓ MLM 损失可视化完成！")

### 6.4 位置编码和段编码详解

**BERT 的三种 Embedding**：

$$\mathbf{E} = \mathbf{E}_{\text{token}} + \mathbf{E}_{\text{position}} + \mathbf{E}_{\text{segment}}$$

#### 位置编码 (Position Embeddings)

**为什么需要位置编码？**
- Transformer 的 self-attention 是置换不变的
- 需要显式编码位置信息

**BERT 的选择**：学习式位置编码

```python
position_embeddings = nn.Embedding(max_position=512, embedding_dim=768)
```

**与 Transformer 原始位置编码的对比**：

| 特性 | Sinusoidal (Transformer) | Learned (BERT) |
|------|-------------------------|----------------|
| 类型 | 固定函数 | 可学习参数 |
| 外推 | 可以 | 不可以 |
| 性能 | 略低 | 略高 |
| 最大长度 | 无限 | 固定 (512) |

#### 段编码 (Segment Embeddings)

**作用**：区分句子 A 和句子 B

```
[CLS] Sentence A [SEP] Sentence B [SEP]
  0      0  0  0   0      1  1  1   1
```

**实现**：

```python
segment_embeddings = nn.Embedding(num_segments=2, embedding_dim=768)
```

**为什么只有 2 个段？**
- BERT 主要处理句子对任务
- 更多段会增加复杂度但收益有限

In [ ]:
# 🔬 Micro Practice: BERT Embeddings 可视化

import matplotlib.pyplot as plt
import numpy as np

def visualize_bert_embeddings():
    """
    可视化 BERT 的三种 embedding
    """
    # 模拟一个句子对
    tokens = ['[CLS]', 'The', 'cat', 'sat', '[SEP]', 'It', 'was', 'happy', '[SEP]']
    seq_len = len(tokens)
    d_model = 8  # 简化的维度

    # 模拟三种 embedding
    np.random.seed(42)
    token_emb = np.random.randn(seq_len, d_model) * 0.5
    position_emb = np.array([np.sin(np.arange(d_model) * i / 10) for i in range(seq_len)])
    segment_emb = np.array([[0.3] * d_model if i < 5 else [-0.3] * d_model for i in range(seq_len)])

    # 最终 embedding
    final_emb = token_emb + position_emb + segment_emb

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Token Embeddings
    im1 = axes[0, 0].imshow(token_emb.T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
    axes[0, 0].set_title('Token Embeddings', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Token Position')
    axes[0, 0].set_ylabel('Embedding Dimension')
    axes[0, 0].set_xticks(range(seq_len))
    axes[0, 0].set_xticklabels(tokens, rotation=45, ha='right')
    plt.colorbar(im1, ax=axes[0, 0])

    # Position Embeddings
    im2 = axes[0, 1].imshow(position_emb.T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
    axes[0, 1].set_title('Position Embeddings', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Token Position')
    axes[0, 1].set_ylabel('Embedding Dimension')
    axes[0, 1].set_xticks(range(seq_len))
    axes[0, 1].set_xticklabels(tokens, rotation=45, ha='right')
    plt.colorbar(im2, ax=axes[0, 1])

    # Segment Embeddings
    im3 = axes[1, 0].imshow(segment_emb.T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
    axes[1, 0].set_title('Segment Embeddings', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Token Position')
    axes[1, 0].set_ylabel('Embedding Dimension')
    axes[1, 0].set_xticks(range(seq_len))
    axes[1, 0].set_xticklabels(tokens, rotation=45, ha='right')

    # 添加段标记
    axes[1, 0].axvline(x=4.5, color='black', linestyle='--', linewidth=2)
    axes[1, 0].text(2, d_model-0.5, 'Segment A', ha='center', fontweight='bold')
    axes[1, 0].text(6.5, d_model-0.5, 'Segment B', ha='center', fontweight='bold')
    plt.colorbar(im3, ax=axes[1, 0])

    # Final Embeddings
    im4 = axes[1, 1].imshow(final_emb.T, cmap='RdBu', aspect='auto', vmin=-2, vmax=2)
    axes[1, 1].set_title('Final Embeddings (Sum of All)', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Token Position')
    axes[1, 1].set_ylabel('Embedding Dimension')
    axes[1, 1].set_xticks(range(seq_len))
    axes[1, 1].set_xticklabels(tokens, rotation=45, ha='right')
    plt.colorbar(im4, ax=axes[1, 1])

    plt.tight_layout()
    plt.show()

    print("\n关键观察:")
    print("  1. Token Embeddings: 编码词汇信息")
    print("  2. Position Embeddings: 编码位置信息（注意周期性模式）")
    print("  3. Segment Embeddings: 区分不同句子（A vs B）")
    print("  4. Final Embeddings: 三者相加，包含完整信息")

visualize_bert_embeddings()
print("\n✓ Embeddings 可视化完成！")

## 5. 完整的 BERT 架构

### 5.1 BERT 组件

```
输入
  ↓
Token Embeddings + Segment Embeddings + Position Embeddings
  ↓
Transformer Encoder × N layers
  ↓
  ├─→ MLM Head (预测掩码 tokens)
  └─→ NSP Head (预测句子关系)
```

### 5.2 特殊 Tokens

- **[CLS]**：句子开始，用于分类任务
- **[SEP]**：句子分隔符
- **[MASK]**：掩码 token
- **[PAD]**：填充 token

### 5.3 BERT 变体

| 模型 | 层数 | 隐藏维度 | 注意力头 | 参数量 |
|------|------|----------|----------|--------|
| BERT-Base | 12 | 768 | 12 | 110M |
| BERT-Large | 24 | 1024 | 16 | 340M |

In [ ]:
# 🔬 Micro Practice: Simplified BERT Model

class SimpleBERT(nn.Module):
    def __init__(self, vocab_size, d_model=768, nhead=12, num_layers=12):
        super().__init__()
        
        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(512, d_model)
        self.segment_embedding = nn.Embedding(2, d_model)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # MLM Head
        self.mlm_head = nn.Linear(d_model, vocab_size)
        
        # NSP Head
        self.nsp_head = nn.Linear(d_model, 2)
    
    def forward(self, input_ids, segment_ids, attention_mask=None):
        # Create position ids
        seq_len = input_ids.size(1)
        position_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        
        # Embeddings
        embeddings = (
            self.token_embedding(input_ids) +
            self.position_embedding(position_ids) +
            self.segment_embedding(segment_ids)
        )
        
        # Encoder
        encoder_output = self.encoder(embeddings, src_key_padding_mask=attention_mask)
        
        # MLM predictions (for all tokens)
        mlm_logits = self.mlm_head(encoder_output)
        
        # NSP prediction (using [CLS] token)
        cls_output = encoder_output[:, 0, :]
        nsp_logits = self.nsp_head(cls_output)
        
        return mlm_logits, nsp_logits

# Test
vocab_size = 30000
model = SimpleBERT(vocab_size, d_model=256, nhead=8, num_layers=4)

# Sample input
batch_size, seq_len = 2, 10
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
segment_ids = torch.zeros(batch_size, seq_len, dtype=torch.long)

mlm_logits, nsp_logits = model(input_ids, segment_ids)

print(f"Input shape: {input_ids.shape}")
print(f"MLM logits shape: {mlm_logits.shape}")
print(f"NSP logits shape: {nsp_logits.shape}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print("\n✓ BERT model implemented!")

## 6.5 高级实践：完整的 BERT 训练和应用

In [ ]:
# 🔬 Micro Practice: 完整 MLM 预训练
# Goal: 在小语料上训练 BERT
# Expected outcome: 看到损失下降，模型学习语言表示

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

class MLMDataset(Dataset):
    """MLM 训练数据集"""
    def __init__(self, texts, vocab_size=1000, max_len=32):
        self.texts = texts
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.mask_token_id = vocab_size
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        # 简化：使用随机 token ids
        tokens = torch.randint(0, self.vocab_size, (self.max_len,))
        labels = tokens.clone()
        
        # 应用 MLM 掩码
        mask_positions = torch.rand(self.max_len) < 0.15
        for i in range(self.max_len):
            if mask_positions[i]:
                rand = torch.rand(1).item()
                if rand < 0.8:
                    tokens[i] = self.mask_token_id
                elif rand < 0.9:
                    tokens[i] = torch.randint(0, self.vocab_size, (1,)).item()
        
        return tokens, labels, mask_positions

# 创建小型 BERT
vocab_size = 1000
model = SimpleBERT(vocab_size + 1, d_model=128, nhead=4, num_layers=2)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# 创建数据集
dataset = MLMDataset(['dummy'] * 100, vocab_size=vocab_size)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# 训练循环
print("开始 MLM 预训练...")
losses = []

for epoch in range(5):
    epoch_loss = 0
    for batch_idx, (tokens, labels, mask_pos) in enumerate(dataloader):
        segment_ids = torch.zeros_like(tokens)
        
        # 前向传播
        mlm_logits, _ = model(tokens, segment_ids)
        
        # 只计算被掩盖位置的损失
        mlm_logits_masked = mlm_logits[mask_pos]
        labels_masked = labels[mask_pos]
        
        loss = criterion(mlm_logits_masked, labels_masked)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/5, Loss: {avg_loss:.4f}")

# 可视化训练过程
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, 6), losses, marker='o', linewidth=2, markersize=8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('MLM Loss', fontsize=12)
plt.title('MLM 预训练损失曲线', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print("\n✓ MLM 预训练完成！")
print(f"最终损失: {losses[-1]:.4f}")
print(f"损失下降: {(losses[0] - losses[-1]) / losses[0] * 100:.1f}%")

In [ ]:
# 🔬 Micro Practice: 注意力权重可视化
# Goal: 看到 BERT 关注什么
# Expected outcome: 注意力热图显示模式

import matplotlib.pyplot as plt
import seaborn as sns
import torch

def visualize_attention_weights():
    """
    可视化 BERT 的注意力权重
    """
    # 创建示例句子
    tokens = ['[CLS]', 'The', 'cat', 'sat', 'on', 'the', 'mat', '[SEP]']
    seq_len = len(tokens)
    
    # 模拟不同层的注意力模式
    np.random.seed(42)
    
    # 第 1 层：局部注意力
    attn_layer1 = np.zeros((seq_len, seq_len))
    for i in range(seq_len):
        for j in range(max(0, i-2), min(seq_len, i+3)):
            attn_layer1[i, j] = np.random.rand() * 0.5 + 0.3
    attn_layer1 = attn_layer1 / attn_layer1.sum(axis=1, keepdims=True)
    
    # 第 6 层：语法注意力
    attn_layer6 = np.random.rand(seq_len, seq_len) * 0.3
    attn_layer6[1, 2] = 0.8  # 'The' -> 'cat'
    attn_layer6[2, 3] = 0.7  # 'cat' -> 'sat'
    attn_layer6[3, 4] = 0.6  # 'sat' -> 'on'
    attn_layer6 = attn_layer6 / attn_layer6.sum(axis=1, keepdims=True)
    
    # 第 12 层：全局注意力
    attn_layer12 = np.random.rand(seq_len, seq_len) * 0.5
    attn_layer12[0, :] = 0.8  # [CLS] 关注所有
    attn_layer12[:, 0] = 0.6  # 所有关注 [CLS]
    attn_layer12 = attn_layer12 / attn_layer12.sum(axis=1, keepdims=True)
    
    # 可视化
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for idx, (attn, layer_name) in enumerate([
        (attn_layer1, '第 1 层\n(局部模式)'),
        (attn_layer6, '第 6 层\n(语法模式)'),
        (attn_layer12, '第 12 层\n(全局模式)')
    ]):
        sns.heatmap(attn, annot=False, cmap='YlOrRd', 
                   xticklabels=tokens, yticklabels=tokens,
                   cbar=True, ax=axes[idx], vmin=0, vmax=1)
        axes[idx].set_title(layer_name, fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Key (被关注的词)')
        axes[idx].set_ylabel('Query (关注的词)')
    
    plt.tight_layout()
    plt.show()
    
    print("\n注意力模式分析:")
    print("  第 1 层: 局部注意力，关注邻近词")
    print("  第 6 层: 语法注意力，关注句法关系")
    print("  第 12 层: 全局注意力，[CLS] 聚合所有信息")

visualize_attention_weights()
print("\n✓ 注意力可视化完成！")

In [ ]:
# 🔬 Micro Practice: 情感分类微调
# Goal: 将 BERT 适配到情感分类任务
# Expected outcome: 在情感数据上达到高准确率

import torch
import torch.nn as nn
import torch.optim as optim

class BERTForSentiment(nn.Module):
    """BERT 情感分类模型"""
    def __init__(self, bert_model, num_labels=2):
        super().__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(256, num_labels)  # d_model=256
    
    def forward(self, input_ids, segment_ids):
        # 获取 BERT 输出
        mlm_logits, _ = self.bert(input_ids, segment_ids)
        
        # 使用 [CLS] token 的表示（第一个位置）
        cls_output = mlm_logits[:, 0, :].mean(dim=-1, keepdim=True)
        cls_output = cls_output.repeat(1, 256)  # 扩展到 d_model
        
        # 分类
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits

# 创建情感分类模型
sentiment_model = BERTForSentiment(model, num_labels=2)
optimizer = optim.Adam(sentiment_model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# 模拟情感数据
print("开始情感分类微调...")
num_samples = 50
X = torch.randint(0, vocab_size, (num_samples, 32))
y = torch.randint(0, 2, (num_samples,))
segment_ids = torch.zeros_like(X)

# 微调循环
losses = []
accuracies = []

for epoch in range(10):
    sentiment_model.train()
    epoch_loss = 0
    correct = 0
    
    for i in range(0, num_samples, 8):
        batch_X = X[i:i+8]
        batch_y = y[i:i+8]
        batch_seg = segment_ids[i:i+8]
        
        # 前向传播
        logits = sentiment_model(batch_X, batch_seg)
        loss = criterion(logits, batch_y)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # 计算准确率
        preds = torch.argmax(logits, dim=1)
        correct += (preds == batch_y).sum().item()
    
    avg_loss = epoch_loss / (num_samples // 8)
    accuracy = correct / num_samples * 100
    losses.append(avg_loss)
    accuracies.append(accuracy)
    
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}/10, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.1f}%")

# 可视化微调过程
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, 11), losses, marker='o', linewidth=2, color='#e74c3c')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('微调损失', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, 11), accuracies, marker='o', linewidth=2, color='#2ecc71')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('分类准确率', fontsize=13, fontweight='bold')
ax2.set_ylim([0, 100])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ 情感分类微调完成！")
print(f"最终准确率: {accuracies[-1]:.1f}%")

In [ ]:
# 🔬 Micro Practice: NER 任务微调
# Goal: Token-level 分类
# Expected outcome: 准确的实体识别

import torch
import torch.nn as nn
import torch.optim as optim

class BERTForNER(nn.Module):
    """BERT NER 模型"""
    def __init__(self, bert_model, num_labels=5):
        super().__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(vocab_size + 1, num_labels)  # Token-level 分类
    
    def forward(self, input_ids, segment_ids):
        # 获取 BERT 输出
        mlm_logits, _ = self.bert(input_ids, segment_ids)
        
        # Token-level 分类
        logits = self.classifier(mlm_logits)
        return logits

# NER 标签：O, B-PER, I-PER, B-LOC, I-LOC
ner_labels = ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC']
num_labels = len(ner_labels)

# 创建 NER 模型
ner_model = BERTForNER(model, num_labels=num_labels)
optimizer = optim.Adam(ner_model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

print("开始 NER 微调...")

# 模拟 NER 数据
num_samples = 40
seq_len = 32
X = torch.randint(0, vocab_size, (num_samples, seq_len))
y = torch.randint(0, num_labels, (num_samples, seq_len))  # Token-level 标签
segment_ids = torch.zeros_like(X)

# 微调循环
losses = []
accuracies = []

for epoch in range(10):
    ner_model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    for i in range(0, num_samples, 8):
        batch_X = X[i:i+8]
        batch_y = y[i:i+8]
        batch_seg = segment_ids[i:i+8]
        
        # 前向传播
        logits = ner_model(batch_X, batch_seg)
        
        # 计算损失（所有 token）
        loss = criterion(logits.view(-1, num_labels), batch_y.view(-1))
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # 计算准确率
        preds = torch.argmax(logits, dim=-1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.numel()
    
    avg_loss = epoch_loss / (num_samples // 8)
    accuracy = correct / total * 100
    losses.append(avg_loss)
    accuracies.append(accuracy)
    
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}/10, Loss: {avg_loss:.4f}, Token Accuracy: {accuracy:.1f}%")

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, 11), losses, marker='o', linewidth=2, color='#e74c3c')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('NER 微调损失', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, 11), accuracies, marker='o', linewidth=2, color='#2ecc71')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Token Accuracy (%)', fontsize=12)
ax2.set_title('NER 准确率', fontsize=13, fontweight='bold')
ax2.set_ylim([0, 100])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ NER 微调完成！")
print(f"最终 Token 准确率: {accuracies[-1]:.1f}%")
print("\nNER 标签:")
for i, label in enumerate(ner_labels):
    print(f"  {i}: {label}")

In [ ]:
# 🔬 Micro Practice: BERT vs GPT 对比
# Goal: 理解何时使用哪个模型
# Expected outcome: 不同任务上的性能差异

import matplotlib.pyplot as plt
import numpy as np

def compare_bert_gpt():
    """
    对比 BERT 和 GPT 在不同任务上的性能
    """
    tasks = ['文本分类', '问答系统', 'NER', '情感分析', '文本生成', '对话生成']
    
    # 模拟性能数据（0-100 分）
    bert_scores = [92, 94, 91, 93, 45, 50]  # BERT 擅长理解任务
    gpt_scores = [85, 82, 83, 86, 95, 94]   # GPT 擅长生成任务
    
    x = np.arange(len(tasks))
    width = 0.35
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 性能对比
    bars1 = ax1.bar(x - width/2, bert_scores, width, label='BERT (双向编码器)',
                    color='#3498db', alpha=0.8)
    bars2 = ax1.bar(x + width/2, gpt_scores, width, label='GPT (单向解码器)',
                    color='#e74c3c', alpha=0.8)
    
    ax1.set_xlabel('任务类型', fontsize=12)
    ax1.set_ylabel('性能得分', fontsize=12)
    ax1.set_title('BERT vs GPT 性能对比', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(tasks, rotation=45, ha='right')
    ax1.legend()
    ax1.set_ylim([0, 100])
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.axhline(y=90, color='green', linestyle='--', alpha=0.3, label='优秀阈值')
    
    # 添加数值标签
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9)
    
    # 优势分析
    categories = ['理解任务\n(分类/QA/NER)', '生成任务\n(文本/对话)']
    bert_avg = [np.mean(bert_scores[:4]), np.mean(bert_scores[4:])]
    gpt_avg = [np.mean(gpt_scores[:4]), np.mean(gpt_scores[4:])]
    
    x2 = np.arange(len(categories))
    bars3 = ax2.bar(x2 - width/2, bert_avg, width, label='BERT',
                    color='#3498db', alpha=0.8)
    bars4 = ax2.bar(x2 + width/2, gpt_avg, width, label='GPT',
                    color='#e74c3c', alpha=0.8)
    
    ax2.set_xlabel('任务类别', fontsize=12)
    ax2.set_ylabel('平均性能', fontsize=12)
    ax2.set_title('任务类别优势对比', fontsize=14, fontweight='bold')
    ax2.set_xticks(x2)
    ax2.set_xticklabels(categories)
    ax2.legend()
    ax2.set_ylim([0, 100])
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 添加数值标签
    for bars in [bars3, bars4]:
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}',
                    ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print("\n关键发现:")
    print("  BERT 优势:")
    print("    ✓ 文本分类、问答、NER、情感分析等理解任务")
    print("    ✓ 需要双向上下文的任务")
    print("    ✓ 判别式任务")
    print("\n  GPT 优势:")
    print("    ✓ 文本生成、对话生成等生成任务")
    print("    ✓ 自回归生成")
    print("    ✓ 生成式任务")
    print("\n  选择建议:")
    print("    - 理解任务 → BERT")
    print("    - 生成任务 → GPT")
    print("    - 两者都需要 → T5 或 BART (Encoder-Decoder)")

compare_bert_gpt()
print("\n✓ BERT vs GPT 对比完成！")

## 7.4 综合项目：构建问答系统

### 项目目标

构建一个基于 BERT 的问答系统，整合所有学到的概念：
- 预训练模型加载
- 任务特定微调
- 推理和评估

### 问答任务说明

**输入**：问题 + 上下文段落
**输出**：答案的起始和结束位置

**示例**：
```
问题: "Who invented the telephone?"
上下文: "Alexander Graham Bell invented the telephone in 1876."
答案: "Alexander Graham Bell" (位置 0-20)
```

In [ ]:
# 🚀 Capstone Project: 问答系统
# 整合所有 BERT 概念：预训练、微调、推理

import torch
import torch.nn as nn
import torch.optim as optim

class BERTForQA(nn.Module):
    """BERT 问答模型"""
    def __init__(self, bert_model):
        super().__init__()
        self.bert = bert_model
        self.qa_outputs = nn.Linear(vocab_size + 1, 2)  # 起始和结束位置
    
    def forward(self, input_ids, segment_ids):
        # 获取 BERT 输出
        mlm_logits, _ = self.bert(input_ids, segment_ids)
        
        # 预测起始和结束位置
        logits = self.qa_outputs(mlm_logits)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1)
        end_logits = end_logits.squeeze(-1)
        
        return start_logits, end_logits

print("=" * 80)
print("构建 BERT 问答系统")
print("=" * 80)

# 1. 创建 QA 模型
print("\n步骤 1: 创建模型")
qa_model = BERTForQA(model)
optimizer = optim.Adam(qa_model.parameters(), lr=3e-5)
criterion = nn.CrossEntropyLoss()

print("✓ 模型创建完成")
print(f"  参数量: {sum(p.numel() for p in qa_model.parameters()):,}")

# 2. 准备数据
print("\n步骤 2: 准备训练数据")
num_samples = 30
seq_len = 64

# 模拟 QA 数据
# 输入: [CLS] 问题 [SEP] 上下文 [SEP]
X = torch.randint(0, vocab_size, (num_samples, seq_len))
segment_ids = torch.cat([
    torch.zeros(num_samples, 20),  # 问题部分
    torch.ones(num_samples, seq_len - 20)  # 上下文部分
], dim=1).long()

# 答案位置（在上下文中）
start_positions = torch.randint(20, 50, (num_samples,))
end_positions = start_positions + torch.randint(1, 10, (num_samples,))
end_positions = torch.clamp(end_positions, max=seq_len-1)

print(f"✓ 数据准备完成")
print(f"  样本数: {num_samples}")
print(f"  序列长度: {seq_len}")

# 3. 微调训练
print("\n步骤 3: 微调训练")
losses = []

for epoch in range(15):
    qa_model.train()
    epoch_loss = 0
    
    for i in range(0, num_samples, 6):
        batch_X = X[i:i+6]
        batch_seg = segment_ids[i:i+6]
        batch_start = start_positions[i:i+6]
        batch_end = end_positions[i:i+6]
        
        # 前向传播
        start_logits, end_logits = qa_model(batch_X, batch_seg)
        
        # 计算损失
        start_loss = criterion(start_logits, batch_start)
        end_loss = criterion(end_logits, batch_end)
        loss = (start_loss + end_loss) / 2
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / (num_samples // 6)
    losses.append(avg_loss)
    
    if (epoch + 1) % 3 == 0:
        print(f"  Epoch {epoch+1}/15, Loss: {avg_loss:.4f}")

print("✓ 训练完成")

# 4. 推理演示
print("\n步骤 4: 推理演示")
qa_model.eval()

with torch.no_grad():
    # 测试样本
    test_input = X[0:1]
    test_seg = segment_ids[0:1]
    
    start_logits, end_logits = qa_model(test_input, test_seg)
    
    # 预测位置
    pred_start = torch.argmax(start_logits, dim=1).item()
    pred_end = torch.argmax(end_logits, dim=1).item()
    
    print(f"\n预测结果:")
    print(f"  起始位置: {pred_start}")
    print(f"  结束位置: {pred_end}")
    print(f"  答案长度: {pred_end - pred_start + 1} tokens")

# 5. 可视化训练过程
print("\n步骤 5: 可视化")

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, 16), losses, marker='o', linewidth=2, color='#3498db')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('QA 模型训练损失', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# 模拟准确率提升
accuracies = [30 + i * 4 + np.random.rand() * 3 for i in range(15)]
plt.plot(range(1, 16), accuracies, marker='o', linewidth=2, color='#2ecc71')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Exact Match (%)', fontsize=12)
plt.title('答案匹配准确率', fontsize=13, fontweight='bold')
plt.ylim([0, 100])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("✓ 问答系统构建完成！")
print("=" * 80)
print("\n项目总结:")
print("  ✓ 模型架构: BERT + QA Head")
print("  ✓ 训练方法: 端到端微调")
print("  ✓ 输出: 答案起始和结束位置")
print("  ✓ 应用场景: 阅读理解、信息抽取")

## 7.5 BERT 变体与改进

### RoBERTa (Robustly Optimized BERT)

**主要改进**：

1. **移除 NSP**：
   - 发现 NSP 任务不是必需的
   - 使用完整句子而非句子对

2. **动态掩码**：
   - 每个 epoch 使用不同的掩码模式
   - 而非静态掩码

3. **更大的批次和数据**：
   - Batch size: 8K (vs BERT 的 256)
   - 训练数据: 160GB (vs BERT 的 16GB)
   - 训练步数: 500K (vs BERT 的 1M)

4. **更长的训练**：
   - 在更多数据上训练更久

**性能提升**：

| 任务 | BERT-Large | RoBERTa-Large | 提升 |
|------|-----------|---------------|------|
| MNLI | 86.6 | 90.2 | +3.6 |
| SQuAD 2.0 | 81.8 | 89.4 | +7.6 |
| RACE | 72.0 | 83.2 | +11.2 |

### ALBERT (A Lite BERT)

**主要优化**：

1. **因式分解嵌入参数**：
   $$E = V \times H \rightarrow E = V \times E + E \times H$$
   - 减少参数：$V \times H$ → $V \times E + E \times H$
   - 当 $E \ll H$ 时，显著减少参数

2. **跨层参数共享**：
   - 所有层共享相同的参数
   - 参数量减少 ~12×

3. **句子顺序预测 (SOP)**：
   - 替代 NSP
   - 预测两个句子的顺序是否正确

**参数对比**：

| 模型 | 参数量 | 性能 |
|------|--------|------|
| BERT-Large | 340M | 基准 |
| ALBERT-xxlarge | 235M | +1.5% |
| ALBERT-xxlarge (共享) | 60M | +1.0% |

### 其他重要变体

**DistilBERT**：
- 知识蒸馏得到的小模型
- 参数量: 66M (BERT-Base 的 60%)
- 性能: 97% of BERT
- 速度: 2× 更快

**ELECTRA**：
- 替换 token 检测而非 MLM
- 更高效的预训练
- 相同计算下性能更好

**DeBERTa**：
- 解耦注意力机制
- 增强的掩码解码器
- SOTA 性能

In [ ]:
# 🔬 Micro Practice: BERT 变体对比

import matplotlib.pyplot as plt
import numpy as np

def compare_bert_variants():
    """
    对比不同 BERT 变体的特性
    """
    models = ['BERT-Base', 'BERT-Large', 'RoBERTa', 'ALBERT', 'DistilBERT', 'ELECTRA']
    
    # 参数量 (百万)
    params = [110, 340, 355, 235, 66, 110]
    
    # 相对性能 (BERT-Base = 100)
    performance = [100, 107, 112, 108, 97, 105]
    
    # 训练速度 (相对)
    speed = [100, 35, 30, 40, 200, 120]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
    
    # 参数量对比
    bars1 = axes[0].bar(models, params, color=colors, alpha=0.8)
    axes[0].set_ylabel('参数量 (百万)', fontsize=11)
    axes[0].set_title('模型参数量', fontsize=12, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    for bar in bars1:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}M',
                    ha='center', va='bottom', fontsize=9)
    
    # 性能对比
    bars2 = axes[1].bar(models, performance, color=colors, alpha=0.8)
    axes[1].set_ylabel('相对性能', fontsize=11)
    axes[1].set_title('模型性能 (BERT-Base=100)', fontsize=12, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].axhline(y=100, color='black', linestyle='--', alpha=0.3)
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].set_ylim([90, 115])
    
    for bar in bars2:
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9)
    
    # 训练速度对比
    bars3 = axes[2].bar(models, speed, color=colors, alpha=0.8)
    axes[2].set_ylabel('相对速度', fontsize=11)
    axes[2].set_title('训练速度 (BERT-Base=100)', fontsize=12, fontweight='bold')
    axes[2].tick_params(axis='x', rotation=45)
    axes[2].axhline(y=100, color='black', linestyle='--', alpha=0.3)
    axes[2].grid(True, alpha=0.3, axis='y')
    
    for bar in bars3:
        height = bar.get_height()
        axes[2].text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("\n模型选择建议:")
    print("  BERT-Base: 标准选择，平衡性能和效率")
    print("  RoBERTa: 追求最佳性能")
    print("  ALBERT: 参数受限场景")
    print("  DistilBERT: 需要快速推理")
    print("  ELECTRA: 预训练效率优先")

compare_bert_variants()
print("\n✓ BERT 变体对比完成！")

## 7.6 使用 Transformers 库实战

### Hugging Face Transformers 简介

**Transformers** 是最流行的预训练模型库，提供：
- 数千个预训练模型
- 统一的 API
- 简化的微调流程
- 生产级性能

### 核心组件

1. **Tokenizer**：文本 → Token IDs
2. **Model**：预训练模型
3. **Trainer**：训练和评估
4. **Pipeline**：端到端推理

In [ ]:
# 🔬 Micro Practice: 加载预训练 BERT

# 注意：这需要安装 transformers 库
# pip install transformers

transformers_demo = '''
# 1. 加载 Tokenizer 和模型
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

print("✓ 模型加载完成")
print(f"模型参数量: {model.num_parameters():,}")

# 2. 文本编码
text = "This movie is amazing!"
encoded = tokenizer(
    text,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

print(f"\n原始文本: {text}")
print(f"Token IDs: {encoded['input_ids']}")
print(f"Attention Mask: {encoded['attention_mask']}")

# 3. 模型推理
import torch

model.eval()
with torch.no_grad():
    outputs = model(**encoded)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

print(f"\n预测结果: {predictions.item()}")
print(f"Logits: {logits}")
'''

print("Transformers 库基础使用示例:")
print("=" * 60)
print(transformers_demo)
print("=" * 60)

print("\n关键步骤:")
print("  1. from_pretrained() 加载预训练模型")
print("  2. tokenizer() 编码文本")
print("  3. model() 前向传播")
print("  4. 解析输出")

print("\n✓ 基础使用演示完成！")

In [ ]:
# 🔬 Micro Practice: 使用 Trainer API 微调

trainer_demo = '''
# 使用 Trainer API 进行微调

from transformers import Trainer, TrainingArguments
from datasets import load_dataset

# 1. 加载数据集
dataset = load_dataset('glue', 'sst2')

# 2. 预处理
def preprocess_function(examples):
    return tokenizer(
        examples['sentence'],
        truncation=True,
        padding=True
    )

tokenized_dataset = dataset.map(preprocess_function, batched=True)

# 3. 配置训练参数
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# 4. 创建 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
)

# 5. 训练
trainer.train()

# 6. 评估
results = trainer.evaluate()
print(f"评估结果: {results}")

# 7. 保存模型
trainer.save_model('./my_bert_model')
tokenizer.save_pretrained('./my_bert_model')
'''

print("Trainer API 微调示例:")
print("=" * 60)
print(trainer_demo)
print("=" * 60)

print("\nTrainer API 优势:")
print("  ✓ 自动处理训练循环")
print("  ✓ 内置评估和日志")
print("  ✓ 支持分布式训练")
print("  ✓ 自动保存检查点")
print("  ✓ 集成 TensorBoard")

print("\n✓ Trainer API 演示完成！")

In [ ]:
# 🔬 Micro Practice: 使用 Pipeline 快速推理

pipeline_demo = '''
# Pipeline: 最简单的使用方式

from transformers import pipeline

# 1. 情感分析
sentiment_analyzer = pipeline('sentiment-analysis')
result = sentiment_analyzer('I love this product!')
print(f"情感分析: {result}")
# 输出: [{'label': 'POSITIVE', 'score': 0.9998}]

# 2. 问答
qa_pipeline = pipeline('question-answering')
context = "Paris is the capital of France."
question = "What is the capital of France?"
result = qa_pipeline(question=question, context=context)
print(f"问答: {result}")
# 输出: {'answer': 'Paris', 'score': 0.99, 'start': 0, 'end': 5}

# 3. 命名实体识别
ner_pipeline = pipeline('ner')
result = ner_pipeline('My name is John and I live in New York.')
print(f"NER: {result}")
# 输出: [{'entity': 'B-PER', 'word': 'John', ...}, ...]

# 4. 文本生成
generator = pipeline('text-generation', model='gpt2')
result = generator('Once upon a time', max_length=30)
print(f"生成: {result}")

# 5. 填空
fill_mask = pipeline('fill-mask', model='bert-base-uncased')
result = fill_mask('Paris is the [MASK] of France.')
print(f"填空: {result}")
# 输出: [{'token_str': 'capital', 'score': 0.99, ...}, ...]
'''

print("Pipeline 快速推理示例:")
print("=" * 60)
print(pipeline_demo)
print("=" * 60)

print("\n支持的 Pipeline 类型:")
pipeline_types = [
    ('sentiment-analysis', '情感分析'),
    ('question-answering', '问答'),
    ('ner', '命名实体识别'),
    ('text-generation', '文本生成'),
    ('fill-mask', '填空'),
    ('summarization', '摘要'),
    ('translation', '翻译'),
    ('zero-shot-classification', '零样本分类')
]

for task, desc in pipeline_types:
    print(f"  • {task:25s} - {desc}")

print("\n✓ Pipeline 演示完成！")

### 模型部署建议

**1. 模型优化**：
```python
# 量化
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased')
model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)
```

**2. ONNX 导出**：
```python
# 导出为 ONNX 格式
from transformers.onnx import export
export(
    preprocessor=tokenizer,
    model=model,
    config=onnx_config,
    opset=13,
    output=Path('model.onnx')
)
```

**3. 使用 FastAPI 部署**：
```python
from fastapi import FastAPI
from transformers import pipeline

app = FastAPI()
classifier = pipeline('sentiment-analysis')

@app.post('/predict')
def predict(text: str):
    result = classifier(text)
    return result
```

**4. 批处理优化**：
```python
# 批量推理
texts = ['text1', 'text2', 'text3']
results = classifier(texts, batch_size=8)
```

## 8. 预训练与微调

### 8.1 预训练过程

**数据**：大规模无标注文本（Wikipedia + BookCorpus）

**目标**：
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{MLM}} + \mathcal{L}_{\text{NSP}}$$

**训练细节**：
- Batch size: 256
- Learning rate: 1e-4 with warmup
- Steps: 1M
- Hardware: 16 TPUs, 4 days

### 8.2 微调范式

1. **加载预训练 BERT**
2. **添加任务特定层**（如分类头）
3. **在标注数据上微调**

**优势**：
- 更少的标注数据
- 更快的收敛
- 更好的性能

### 8.3 使用 Transformers 库

```python
from transformers import BertTokenizer, BertForSequenceClassification

# 加载预训练模型
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# 微调
# ... training code ...
```

## 9. 总结与展望

### 核心要点

1. **双向编码**：BERT 通过 MLM 实现真正的双向理解
2. **预训练目标**：MLM + NSP 学习通用语言表示
3. **微调范式**：预训练 + 微调成为 NLP 标准流程
4. **影响深远**：BERT 开启了预训练模型时代

### 最佳实践

**选择合适的 BERT 变体**：
- 标准任务 → BERT-Base
- 追求性能 → RoBERTa / BERT-Large
- 资源受限 → DistilBERT / ALBERT
- 快速实验 → 使用 Transformers Pipeline

**微调技巧**：
1. **学习率**：2e-5 到 5e-5（比预训练小）
2. **Warmup**：前 10% 步数线性增加学习率
3. **批次大小**：16 或 32
4. **Epoch 数**：2-4 个 epoch 通常足够
5. **梯度裁剪**：防止梯度爆炸

**数据需求**：
- 文本分类：1K-10K 样本
- NER：5K-20K 样本
- 问答：10K-100K 样本
- 领域适应：可以从少量数据开始

### 常见问题 (FAQ)

**Q1: 为什么用 MLM 而不是标准 LM？**
- 标准 LM 是单向的，MLM 允许双向上下文
- 双向上下文对理解任务至关重要

**Q2: NSP 真的必要吗？**
- 后续研究（RoBERTa）表明 NSP 可能不是必需的
- 但对句子对任务仍有帮助

**Q3: BERT vs GPT 如何选择？**
- BERT：理解任务（分类、QA、NER）
- GPT：生成任务（文本生成、对话）
- T5/BART：两者都需要

**Q4: 如何选择 BERT 变体？**
- 看任务需求和资源限制
- 参考上面的"选择合适的 BERT 变体"

**Q5: 微调需要多少数据？**
- 取决于任务复杂度
- 一般来说：分类 < NER < QA
- 可以从小数据集开始，逐步增加

**Q6: 如何处理长文本？**
- 截断：取前 512 tokens
- 滑动窗口：重叠处理多个片段
- Longformer/BigBird：支持更长序列

**Q7: BERT 的计算成本如何？**
- 预训练：非常昂贵（数千 GPU 小时）
- 微调：可接受（几个 GPU 小时）
- 推理：中等（可以优化）

**Q8: 如何加速 BERT 推理？**
- 量化：INT8 量化
- 蒸馏：DistilBERT
- 剪枝：移除不重要的权重
- ONNX：优化的运行时

**Q9: BERT 适合哪些语言？**
- 原始 BERT：英语
- mBERT：104 种语言
- XLM-RoBERTa：100 种语言
- 各语言专用 BERT

**Q10: 如何评估 BERT 模型？**
- 使用任务特定指标
- 在验证集上评估
- 考虑推理速度
- 进行错误分析

### 实践建议

**从简单开始**：
1. 使用 Transformers Pipeline 快速验证
2. 在小数据集上微调
3. 逐步增加复杂度
4. 优化和部署

**调试技巧**：
1. 先在小数据上过拟合
2. 检查数据预处理
3. 监控训练曲线
4. 使用梯度检查

**性能优化**：
1. 混合精度训练（FP16）
2. 梯度累积
3. 分布式训练
4. 模型并行

### 下一章预告

**Module 4.3: GPT Architecture**
- 自回归语言模型
- 生成式预训练
- GPT-2 和 GPT-3
- BERT vs GPT 深入对比

### 💡 思考题

1. 为什么 BERT 不能直接用于文本生成？
   - 提示：考虑训练目标和注意力机制

2. 如何修改 BERT 来处理更长的序列？
   - 提示：位置编码、注意力机制

3. BERT 的哪些设计选择对性能影响最大？
   - 提示：双向编码、预训练数据、模型大小

4. 如何将 BERT 应用于多语言任务？
   - 提示：mBERT、XLM-RoBERTa

5. 在资源受限的情况下如何使用 BERT？
   - 提示：蒸馏、量化、剪枝

### 参考资源

**论文**：
- [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805)
- [RoBERTa: A Robustly Optimized BERT Pretraining Approach](https://arxiv.org/abs/1907.11692)
- [ALBERT: A Lite BERT](https://arxiv.org/abs/1909.11942)

**代码**：
- [Hugging Face Transformers](https://github.com/huggingface/transformers)
- [Google BERT](https://github.com/google-research/bert)

**教程**：
- [Hugging Face Course](https://huggingface.co/course)
- [Jay Alammar's Blog](http://jalammar.github.io/)

**工具**：
- [Weights & Biases](https://wandb.ai/)
- [TensorBoard](https://www.tensorflow.org/tensorboard)

---

## 🎉 恭喜完成 Module 4.2！

你已经掌握了：
- ✅ BERT 的动机和创新
- ✅ 双向编码的重要性
- ✅ MLM 和 NSP 预训练目标
- ✅ BERT 架构和实现
- ✅ 微调方法和最佳实践
- ✅ BERT 变体和改进
- ✅ Transformers 库实战
- ✅ 10+ 个微型实践项目
- ✅ 1 个综合问答系统项目

**下一步**：
- 学习 GPT 架构（Module 4.3）
- 探索更多预训练模型
- 在实际项目中应用 BERT

继续加油！🚀

## 思考题参考答案

### 问题 1：为什么双向上下文在语言理解任务中如此重要？单向模型有哪些局限性？

**双向上下文的重要性**

自然语言的语义理解高度依赖上下文，一个词的含义往往需要同时考虑其前后文才能确定：

**例子：歧义消解**
```
句子："我去银行取钱"
→ 单向（只看前文）：看到"银行"时，前文只有"我去"，无法确定语义
→ 双向（前后文）：结合后文"取钱"，明确是金融机构而非河岸
```

**单向模型的局限性**

| 问题类型 | 单向模型的局限 | 双向模型的优势 |
|---------|--------------|--------------|
| 词义歧义 | 只能用前文消歧，准确率低 | 利用完整上下文，准确消歧 |
| 指代消解 | 无法知道后文出现的被指代词 | 同时看到代词和被指代词 |
| 情感分析 | 可能忽略句末的否定词 | 全局理解句子语义 |
| 命名实体识别 | 实体边界依赖双侧上下文 | 充分利用实体周围信息 |
| 阅读理解 | 无法有效关联问题与答案 | 问题和段落的双向对齐 |

**数学角度**

单向模型的表示：$h_i = f(w_1, w_2, \ldots, w_i)$（只依赖左侧词）

双向模型的表示：$h_i = f(w_1, \ldots, w_i, \ldots, w_n)$（依赖所有位置的词）

BERT 通过掩码语言建模（MLM）在不泄露目标词信息的前提下实现了真正的双向上下文建模。

---

### 问题 2：BERT 如何通过掩码语言模型（MLM）实现双向编码？为什么不能直接用普通语言模型？

**普通语言模型不能直接双向的原因**

传统的自回归语言模型（如 GPT）需要从左到右预测，如果允许双向访问，模型训练时会看到要预测的词本身，导致"信息泄露"：

$$P(w_i | w_1, \ldots, w_{i-1}, \mathbf{w_i}, w_{i+1}, \ldots, w_n) = 1 \quad \text{（退化，无需学习）}$$

**MLM 的解决方案**

BERT 的 MLM 策略：
1. 随机选择 15% 的 token
2. 对选中的 token 执行替换：
   - 80%：替换为 `[MASK]`
   - 10%：替换为随机词
   - 10%：保持原词不变
3. 训练模型预测被遮掩位置的原始词

**为什么这样设计？**

| 比例 | 原因 |
|------|------|
| 80% `[MASK]` | 迫使模型利用上下文预测，学习双向依赖 |
| 10% 随机词 | 防止模型只关注 `[MASK]` 标记，增加鲁棒性 |
| 10% 原词 | 消除微调时没有 `[MASK]` 的训练-推理差异（预训练偏差） |

**关键公式**

MLM 损失只对被遮掩的位置计算：

$$\mathcal{L}_{\text{MLM}} = -\sum_{i \in \mathcal{M}} \log P(w_i \mid \hat{w}_1, \ldots, \hat{w}_n)$$

其中 $\mathcal{M}$ 是被遮掩位置的集合，$\hat{w}$ 是经过替换后的输入序列。

**双向实现机制**

在自注意力计算中，每个位置的 `[MASK]` token 可以与**所有其他位置**（包括其两侧）进行注意力交互：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

没有因果掩码，因此真正实现了全局双向上下文。

---

### 问题 3：BERT 与 GPT 有什么本质区别？各自适合哪类任务？

**架构对比**

| 特性 | BERT | GPT |
|------|------|-----|
| **架构类型** | Encoder-only | Decoder-only |
| **注意力方向** | 双向（完全自注意力） | 单向（因果注意力，只看左侧） |
| **预训练任务** | MLM + NSP | 自回归语言建模（下一词预测） |
| **特殊 token** | `[CLS]`, `[SEP]`, `[MASK]` | `<\|endoftext\|>` |
| **位置编码** | 可学习的绝对位置编码 | 可学习的绝对位置编码 |
| **输出形式** | 每个 token 的上下文表示 | 每个位置的下一词概率分布 |
| **微调方式** | 需要任务特定的微调头 | 支持 few-shot，也可微调 |

**核心设计哲学差异**

```
BERT：先双向理解全局，再做预测
  "The bank [MASK] the river" → 利用"the river"判断[MASK]='of'

GPT：从左到右逐步生成
  "The bank of" → 生成"the" → 生成"river"
```

**任务适配性**

| 任务类型 | 推荐模型 | 原因 |
|---------|---------|------|
| 文本分类（情感/主题） | BERT | `[CLS]` 聚合全局语义，双向理解 |
| 命名实体识别（NER） | BERT | 每个 token 的双向上下文表示 |
| 问答（QA） | BERT | 同时理解问题和文档，精准定位 |
| 文本蕴含（NLI） | BERT | 句对关系需要双向理解 |
| 文本生成 | GPT | 自回归设计天然支持顺序生成 |
| 对话系统 | GPT | 多轮上下文的自然延续 |
| 代码生成 | GPT | 代码是序列化结构，适合逐步生成 |
| Few-shot 学习 | GPT | 上下文学习（in-context learning）能力强 |

**选择建议**

- 任务需要**理解和分析**现有文本 → 优选 BERT（或其变体 RoBERTa、DeBERTa）
- 任务需要**生成和创作**新文本 → 优选 GPT
- 资源受限需要快速部署 → BERT（推理快，易于微调）
- 零样本/少样本场景 → GPT（更强的迁移能力）

**近年新趋势**

T5、BART 等 Encoder-Decoder 模型尝试融合两者优势；而 GPT-4 等超大规模模型通过规模效应在理解任务上也超越了 BERT，使得"理解用 BERT、生成用 GPT"的界限逐渐模糊。